In [1]:
from pathlib import Path
import pandas as pd
import torchvision.models as models
from sklearn.model_selection import train_test_split
import torch
from torchvision import transforms
from PIL import Image
import torch.nn as nn
from tqdm import tqdm

In [2]:
DATA_DIR = Path(r"data\PlantVillage-Dataset\raw\color")
print(f"  Path : {DATA_DIR}")
print(f"  Existe : {DATA_DIR.exists()}")

if DATA_DIR.exists():
    classes = sorted([
        folder.name
        for folder in DATA_DIR.iterdir()
        if folder.is_dir()
    ])
    
    print(f"\n Classes trouvées : {len(classes)}")
    for i, cls in enumerate(classes[:20], 1): 
        print(f"  {i:2d}. {cls}")
    image_extensions = {".jpg", ".jpeg", ".png"}
    class_counts = {}
    
    for cls in classes:
        count = sum(
            1
            for f in (DATA_DIR / cls).iterdir()
            if f.is_file() and f.suffix.lower() in image_extensions
        )
        class_counts[cls] = count
    
    print(f"\n Images par classe :")
    for cls, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {count:5d} - {cls}")
    
    total = sum(class_counts.values())
    print(f"\nTotal : {total} images")

  Path : data\PlantVillage-Dataset\raw\color
  Existe : True

 Classes trouvées : 38
   1. Apple___Apple_scab
   2. Apple___Black_rot
   3. Apple___Cedar_apple_rust
   4. Apple___healthy
   5. Blueberry___healthy
   6. Cherry_(including_sour)___Powdery_mildew
   7. Cherry_(including_sour)___healthy
   8. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
   9. Corn_(maize)___Common_rust_
  10. Corn_(maize)___Northern_Leaf_Blight
  11. Corn_(maize)___healthy
  12. Grape___Black_rot
  13. Grape___Esca_(Black_Measles)
  14. Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
  15. Grape___healthy
  16. Orange___Haunglongbing_(Citrus_greening)
  17. Peach___Bacterial_spot
  18. Peach___healthy
  19. Pepper,_bell___Bacterial_spot
  20. Pepper,_bell___healthy

 Images par classe :
   5507 - Orange___Haunglongbing_(Citrus_greening)
   5357 - Tomato___Tomato_Yellow_Leaf_Curl_Virus
   5090 - Soybean___healthy
   2297 - Peach___Bacterial_spot
   2127 - Tomato___Bacterial_spot
   1909 - Tomato___Late_blig

In [3]:
image_extensions = {".jpg", ".jpeg", ".png"}
data = []

for cls in classes:
    class_dir = DATA_DIR / cls
    image_files = [
        f for f in class_dir.iterdir()
        if f.is_file() and f.suffix.lower() in image_extensions
    ]
    
    for img_file in image_files:
        data.append({
            "class": cls,
            "filename": img_file.name,
            "full_path": str(img_file)
        })
df_all = pd.DataFrame(data)

In [4]:
df_train_temp, df_val_test = train_test_split(
    df_all,
    test_size=0.20,
    random_state=42,
    stratify=df_all["class"]
)

df_val, df_test = train_test_split(
    df_val_test,
    test_size=0.50,
    random_state=42,
    stratify=df_val_test["class"]
)

df_train = df_train_temp.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
print(f" Distribution :")
print(f"  TRAIN : {len(df_train)} images ({len(df_train)/len(df_all)*100:.1f}%)")
print(f"  VAL   : {len(df_val)} images ({len(df_val)/len(df_all)*100:.1f}%)")
print(f"  TEST  : {len(df_test)} images ({len(df_test)/len(df_all)*100:.1f}%)")
print(f"  TOTAL : {len(df_train) + len(df_val) + len(df_test)}")

 Distribution :
  TRAIN : 43444 images (80.0%)
  VAL   : 5430 images (10.0%)
  TEST  : 5431 images (10.0%)
  TOTAL : 54305


In [ ]:


class PlantVillageDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.classes = sorted(df['class'].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.idx_to_class = {idx: cls for cls, idx in self.class_to_idx.items()}
        self.num_classes = len(self.classes)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row['full_path']
        class_name = row['class']
        image = Image.open(image_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        label = self.class_to_idx[class_name]  
        return image, label
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset_train = PlantVillageDataset(df_train, transform=train_transform)
dataset_val = PlantVillageDataset(df_val, transform=val_test_transform)
dataset_test = PlantVillageDataset(df_test, transform=val_test_transform)

BATCH_SIZE = 4

loader_train = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4 
)

loader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

images, labels = next(iter(loader_train))
print(f"  Batch shape : {images.shape}")
print(f"  Labels shape : {labels.shape}")

In [ ]:
model_pretrained = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
total_params = sum(p.numel() for p in model_pretrained.parameters())
trainable_params = sum(p.numel() for p in model_pretrained.parameters() if p.requires_grad)

print(f"\n Paramètres :")
print(f"  Total : {total_params:,}")
print(f"  Trainable : {trainable_params:,}")

print(f"\n Dernière couche :")
print(model_pretrained.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 128MB/s] 



 Paramètres :
  Total : 5,288,548
  Trainable : 5,288,548

 Dernière couche :
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [ ]:
device = 'cuda'
num_classes = 38
model_pretrained.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(1280, 128),
    nn.SiLU(),
    nn.Linear(128, num_classes)
)
model_pretrained = model_pretrained.to(device)
print(model_pretrained.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1280, out_features=128, bias=True)
  (2): SiLU()
  (3): Linear(in_features=128, out_features=38, bias=True)
)


In [ ]:
for param in model_pretrained.features.parameters():
    param.requires_grad = False
total_params = sum(p.numel() for p in model_pretrained.parameters())
trainable_params = sum(p.numel() for p in model_pretrained.parameters() if p.requires_grad)

In [ ]:
class TrainerEfficientNet:
    def __init__(self, model, device, learning_rate=0.01):
        self.model = model
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate
        )
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, 
            mode='min', 
            factor=0.5, 
            patience=2
        )
        self.history = {
            "train_loss": [],
            "train_acc": [],
            "val_loss": [],
            "val_acc": []
        }
        self.best_val_acc = 0
        self.best_model_path = None
    
    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for images, labels in tqdm(loader, desc="Training"):
            images, labels = images.to(self.device), labels.to(self.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        
        avg_loss = total_loss / len(loader)
        accuracy = correct / total * 100
        return avg_loss, accuracy
    
    def validate(self, loader):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(loader, desc="Validation", leave=False):
                images, labels = images.to(self.device), labels.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        
        avg_loss = total_loss / len(loader)
        accuracy = correct / total * 100
        return avg_loss, accuracy
    
    def train(self, train_loader, val_loader, epochs=10):
        print(f"\n Fine-tuning EfficientNet-B0 sur {epochs} époques\n")
        
        for epoch in range(epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.validate(val_loader)
            
            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)
            
            self.scheduler.step(val_loss)
            
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.best_model_path = f"models/efficientnet_b0_finetuned_epoch_{epoch+1}.pth"
                torch.save(self.model.state_dict(), self.best_model_path)
                marker = "  best"
            else:
                marker = ""
            
            print(f"Epoch [{epoch+1:2d}/{epochs}] | TRAIN Loss={train_loss:.4f} Acc={train_acc:.2f}% | VAL Loss={val_loss:.4f} Acc={val_acc:.2f}%{marker}")
    
    def get_history(self):
        return self.history


In [ ]:

models_dir = Path("/kaggle/working/models")
models_dir.mkdir(exist_ok=True)

print(f"Dossier créé : {models_dir}")
trainer_eff = TrainerEfficientNet(model_pretrained, device, learning_rate=0.005)
trainer_eff.train(
    train_loader=loader_train,
    val_loader=loader_val,
    epochs=5
)

history_eff = trainer_eff.get_history()
print("\n Fine-tuning terminé !")
print(f"  Meilleur modèle : {trainer_eff.best_model_path}")
print(f"  Meilleure accuracy VAL : {trainer_eff.best_val_acc:.2f}%")

Dossier créé : /kaggle/working/models

 Fine-tuning EfficientNet-B0 sur 5 époques



Training: 100%|██████████| 1358/1358 [02:28<00:00,  9.13it/s]


Epoch [ 1/5] | TRAIN Loss=0.5163 Acc=84.29% | VAL Loss=0.2410 Acc=92.21%  best


Training: 100%|██████████| 1358/1358 [01:53<00:00, 11.98it/s]


Epoch [ 2/5] | TRAIN Loss=0.3831 Acc=88.53% | VAL Loss=0.2055 Acc=93.24%  best


Validation:  65%|██████▌   | 111/170 [00:06<00:03, 18.53it/s]

In [ ]:
def evaluate_on_test(model, loader, device, model_name):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Evaluating {model_name}", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total * 100
    return accuracy

model_pretrained.load_state_dict(
    torch.load(trainer_eff.best_model_path)
)
test_acc_eff = evaluate_on_test(model_pretrained, loader_test, device, "EfficientNet-B0")

print(f"\nTEST Set Results :")
print(f"  EfficientNet-B0 : {test_acc_eff:.2f}%")